In [1]:
!pip install -U transformers accelerate datasets fsspec aiohttp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
from datasets import load_dataset
multi_news = load_dataset("multi_news", split='test')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

multi_news.py: 0.00B [00:00, ?B/s]

The repository for multi_news contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/multi_news.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


train.src.cleaned:   0%|          | 0.00/548M [00:00<?, ?B/s]

...

test.src.cleaned:   0%|          | 0.00/69.0M [00:00<?, ?B/s]

test.tgt:   0%|          | 0.00/7.31M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44972 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5622 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5622 [00:00<?, ? examples/s]

In [3]:
multi_news.to_pandas()

,document,summary
0,GOP Eyes Gains As Voters In 11 States Pick Gov...,– It's a race for the governor's mansion in 11...
1,\n \n \n \n UPDATE: 4/19/2001 Read Richard Met...,– It turns out Facebook is only guilty of abou...
2,It's the Golden State's latest version of the ...,– Not a big fan of Southern California? Neithe...
3,The seed for this crawl was a list of every ho...,– Why did Microsoft buy Nokia's phone business...
4,After a year in which liberals scored impressi...,– The Supreme Court is facing a docket of high...
...,...,...
5617,Tweet with a location \n \n You can add locati...,– The traditional end-of-summit group photo at...
5618,Loic Venance/AFP/Getty Images \n \n The awards...,– Sofia Coppola scored a historic victory at t...
5619,(CNN) A federal criminal investigation into a ...,– The duck boat sinking that killed 17 on a Mi...
5620,An archive of the public statements deleted by...,– Note to tweeting politicians: Watch what you...


In [4]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained('t5-small')

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [5]:
multi_news=multi_news.train_test_split(test_size= .2)

In [6]:
prefix='summarize'

def process_function(examples):
  inputs=[prefix+doc for doc in examples['document']]
  model_inputs=tokenizer(inputs,max_length=1024,truncation=True)
  labels=tokenizer(text=examples['summary'],max_length=128,truncation=True)
  model_inputs['labels']=labels['input_ids']

  return model_inputs


In [7]:
tokenized_multi_news = multi_news.map(process_function, batched=True)


Map:   0%|          | 0/4497 [00:00<?, ? examples/s]

Map:   0%|          | 0/1125 [00:00<?, ? examples/s]

In [8]:
from transformers import DataCollatorForSeq2Seq, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,model='t5-small')
model=AutoModelForSeq2SeqLM.from_pretrained('t5-small')


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
from transformers import Seq2SeqTrainingArguments

trainings_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    fp16=True,
    report_to="none" # Disable reporting to Weights & Biases
)

In [10]:
trainer=Seq2SeqTrainer(
    model=model,
    args=trainings_args,
    train_dataset=tokenized_multi_news['train'],
    eval_dataset=tokenized_multi_news['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/tmp/ipython-input-10-3924138155.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer=Seq2SeqTrainer(


In [11]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,No log,2.859235
2,3.237100,2.787467
3,3.237100,2.755903
4,2.997200,2.738116
5,2.997200,2.725202
6,2.937600,2.717620
7,2.937600,2.712914
8,2.931400,2.708443
9,2.912400,2.707153
10,2.912400,2.706177


TrainOutput(global_step=2820, training_loss=2.9914106003781584, metrics={'train_runtime': 2723.3388, 'train_samples_per_second': 16.513, 'train_steps_per_second': 1.035, 'total_flos': 1.2172386272477184e+16, 'train_loss': 2.9914106003781584, 'epoch': 10.0})

In [14]:
text='''
The Transformer architecture is a deep learning model that processes input sequences in parallel using self-attention to understand the context between words. It is widely used in Natural Language Processing (NLP) and vision, speech, and multi-modal tasks. The architecture consists of an encoder and decoder structure, with each layer having multi-head self-attention, feed forward neural network (FFN), residual connections, and layer normalization. The core components of Transformers include self-attention, multi-head attention, positional encoding, feed-forward layer, and layer normalization and residuals. The training details include cross entropy loss, optimizer, and teacher forcing. Transformers are used in machine translation, text summarization, chatbots, code generation, and vision transformers. They also have applications in visualization.
'''

In [15]:
input_ids=tokenizer(text,max_length=1024,truncation=True,return_tensors='pt').input_ids
input_ids=input_ids.to('cuda')

In [17]:
import torch
with torch.no_grad():
  if model.device.type=='cuda':
    output=model.generate(input_ids,max_length=128,num_beams=5)
  else:
    # If not on GPU, use the model on CPU
    output=model.generate(input_ids.to(model.device),max_length=128,num_beams=5)


summary_ids=output[0].tolist()
summary=tokenizer.decode(summary_ids,skip_special_tokens=True)
print(summary)

– The Transformer architecture is a deep learning model that processes input sequences in parallel using self-attention to understand the context between words. The architecture consists of an encoder and decoder structure, with each layer having multi-head self-attention, feed forward neural network (FFN), residual connections, and layer normalization and residuals. The core components of Transformers include self-attention, multi-head attention, positional encoding, feed-forward layer, and layer normalization and residuals. The core components of Transformers


In [21]:
ref_summary="""
The Transformer architecture is a deep learning model that processes input sequences in parallel using self-attention to understand the context between words. It is widely used in Natural Language Processing (NLP) and vision, speech, and multi-modal tasks. The architecture consists of an encoder and decoder structure, with each layer having multi-head self-attention, feed forward neural network (FFN), residual connections, and layer normalization. The core components of Transformers include self-attention, multi-head attention, positional encoding, feed-forward layer, and layer normalization and residuals. The training details include cross entropy loss, optimizer, and teacher forcing. Transformers are used in machine translation, text summarization, chatbots, code generation, and vision transformers. They also have applications in visualization.
"""

In [19]:
!pip install rouge

In [22]:
from rouge import Rouge
rouge=Rouge()
scores=rouge.get_scores(summary,ref_summary)
scores

[{'rouge-1': {'r': 0.6022727272727273,
   'p': 0.9814814814814815,
   'f': 0.7464788685260861},
  'rouge-2': {'r': 0.5648148148148148,
   'p': 0.9682539682539683,
   'f': 0.7134502877439213},
  'rouge-l': {'r': 0.6022727272727273,
   'p': 0.9814814814814815,
   'f': 0.7464788685260861}}]

In [23]:
trainer.save_model()

In [24]:
model.save_pretrained("summarizer")